In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


This notebook calculates the nutrients, impacts and cost per gram for all meat and dairy ingredients that are used to calculate the impact of reducing each meat and dairy ingredient in the simulation.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

data_path = Path("data")

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from analysis_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


In [ ]:
from data_processing import item_only_nutrients, unique_mixed_items, add_env_data_description, add_dairy_ndb

# Data preprocessing

In [ ]:
df_impacts = pd.read_parquet(data_path / 'df_impacts_per_100g_1902.parquet')

In [ ]:
diet_data_path = data_path / 'diet_data.parquet'
diet_data = pd.read_parquet(diet_data_path)

matched_items_data_path = data_path / "foodDB_data_processing/NDB_matching_data_FINAL.csv"
matched_items_data = pd.read_csv(matched_items_data_path)

ndb_data_path = data_path / 'NDB_data/NDB_intake24-nutrient-mapping-UK_V2_2022-18-10-2022.csv'
ndb_data = pd.read_csv(ndb_data_path)

# 2023 NDB data used for nutrient composition of some dairy ingredients in the FSA recipeDB: NOT PUBLICALLY AVAILABLE
ndb_comprehensive = pd.read_excel(data_path / 'NDB_data/UK_NDB_1.2 2023.09.20_comprehensive.xlsx')

dairy_per100g = pd.read_excel(data_path / 'dairy_disag/dairy_disag_16122023.xlsx')

In [ ]:
with open(data_path / 'mappings/ndb_columns_rename.json', 'r') as fp:
    ndb2022_cols_rename = json.load(fp)

In [ ]:
with open(data_path / 'mappings/ndb_comprehensive_nutrient_map.json', 'r') as fp:
    ndb2023_cols_rename = json.load(fp)

ndb2023_cols_rename = {ndb_2023_nutr: nutr for nutr, ndb_2023_nutr in ndb2023_cols_rename.items()}

In [ ]:
ndb_data.rename(columns=ndb2022_cols_rename, inplace=True)

In [ ]:
ndb_comprehensive.rename(columns=ndb2023_cols_rename, inplace=True)

In [ ]:
nutrients = np.loadtxt(data_path / 'indicator_lists/nutrients.txt', dtype=str).tolist()
env_columns = np.loadtxt(data_path / 'indicator_lists/env_columns.txt', dtype=str).tolist()
mean_env_columns = np.loadtxt(data_path / 'indicator_lists/mean_env_columns.txt', dtype=str).tolist()
error_columns = np.loadtxt(data_path / 'indicator_lists/error_columns.txt', dtype=str).tolist()
price_columns = np.loadtxt(data_path / 'indicator_lists/price_columns.txt', dtype=str).tolist()
dairy_categories = np.loadtxt(data_path / 'indicator_lists/food_groups_dairy.txt', dtype=str).tolist()

In [ ]:
ndb_data.loc[:, dairy_categories]=0
ndb_data.loc[:, 'TotalGrams']=100

ndb_comprehensive.loc[:, dairy_categories]=0
ndb_comprehensive.loc[:, 'TotalGrams']=100

Add the disaggregated dairy data to the NDB dataset

In [ ]:
for cat in dairy_categories:
  ndb_data = ndb_data.apply(lambda row: add_dairy_ndb(row, dairy_category=cat, dairy_per100g=dairy_per100g), axis=1)
  ndb_comprehensive = ndb_comprehensive.apply(lambda row: add_dairy_ndb(row, dairy_category=cat, dairy_per100g=dairy_per100g), axis=1)

# Compute the nutrients and impacts per gram of each meat ingredient, based on item description in SHeS

In [ ]:
lamb_only, mixed_lamb = unique_mixed_items(variable='Lambg', diet_data=diet_data)
beef_only, mixed_beef = unique_mixed_items(variable='Beefg', diet_data=diet_data)
pork_only, mixed_pork = unique_mixed_items(variable='Porkg', diet_data=diet_data)
processedredmeat_only, processedredmeat_mixed = unique_mixed_items(variable='ProcessedRedMeatg', diet_data=diet_data)
otherredmeat_only, otherredmeat_mixed = unique_mixed_items(variable='OtherRedMeatg', diet_data=diet_data)
burgers_only, burgers_mixed = unique_mixed_items(variable='Burgersg', diet_data=diet_data)
sausages_only, sausages_mixed = unique_mixed_items(variable='Sausagesg', diet_data=diet_data)
offal_only, offal_mixed = unique_mixed_items(variable='Offalg', diet_data=diet_data)
poultry_only, poultry_mixed = unique_mixed_items(variable='Poultryg', diet_data=diet_data)
processedpoultry_only, processedpoultry_mixed = unique_mixed_items(variable='ProcessedPoultryg', diet_data=diet_data)
gamebirds_only, gamebirds_mixed = unique_mixed_items(variable='GameBirdsg', diet_data=diet_data)

In [ ]:
nutrients_per_gram_meat = item_only_nutrients(item_only=beef_only, indicators=nutrients, diet_data=diet_data)

for item in [lamb_only, pork_only, processedredmeat_only, otherredmeat_only, burgers_only, sausages_only, offal_only, poultry_only, processedpoultry_only, gamebirds_only]:
  df_new = item_only_nutrients(item_only=item, indicators=nutrients, diet_data=diet_data)
  nutrients_per_gram_meat = pd.concat([nutrients_per_gram_meat, df_new], ignore_index=False)

Add the foodDB indicators per gram

In [ ]:
nutrients_per_gram_meat = nutrients_per_gram_meat.apply(lambda row: add_env_data_description(row,
                                                                                             df_impacts=df_impacts,
                                                                                             diet_data=diet_data,
                                                                                             matched_items_data = matched_items_data,
                                                                                             env_columns=env_columns,
                                                                                             mean_env_columns=mean_env_columns,
                                                                                             error_columns=error_columns,
                                                                                             ),
                                                        axis=1)

Three items did not have environmental impact data matched based on description and another did not have price data. Include the environmental impacts based on food code and the missing price data based on a similar item.

In [ ]:
missing_meat_items = {'Pork chop/loin, grilled, fat not eaten': 1024,
                      'Pork chop/loin, grilled, fat eaten': 1026,
                      'Bacon, streaky, smoked, grilled': 8244
                      }

for desc, matched_fc in missing_meat_items.items():
  descriptions_ndb = matched_items_data[matched_items_data['Food composition record ID']==matched_fc]['Local description'].tolist()
  env_data_mean = df_impacts.loc[descriptions_ndb, mean_env_columns]/100
  env_data_mean = env_data_mean.mean(axis=0)
  env_data_error = df_impacts.loc[descriptions_ndb, error_columns]/100
  env_data_error = (env_data_error**2).sum(axis=0)
  env_data_error = np.sqrt(env_data_error)/len(descriptions_ndb)
  env_data = pd.concat([env_data_mean, env_data_error], axis=0)
  env_data = env_data[env_columns].to_numpy()

  nutrients_per_gram_meat.loc[desc, env_columns] = env_data

# No price data for this iet
nutrients_per_gram_meat.loc['Beef burger, 100% beef, grilled (no bun)', price_columns] =  nutrients_per_gram_meat.loc['Beef burger, grilled (no bun)', price_columns]

In [ ]:
# Check that all values have been added to the dataset as floats
for name, value in nutrients_per_gram_meat['median_GHG'].items():
  if not isinstance(value, float):
    print(name)

In [ ]:
missing_nutrient_items = nutrients_per_gram_meat[nutrients_per_gram_meat['Proteing'].isna()].index.tolist()

In [ ]:
for item in missing_nutrient_items:
  nutrients_per_gram_meat.loc[item, nutrients] = ndb_data[ndb_data['Local description']==item][nutrients].to_numpy()/100

In [ ]:
# Save the dataset
nutrients_per_gram_meat.to_parquet(data_path / '/nutrients_per_gram_meat.parquet')

# Compute the nutrients and impact of each dairy ingredient, based on the food code

In [ ]:
from data_processing import add_env_data_food_code, item_only_nutrients_dairy

In [ ]:
# Load dictionary of composite and non-composite dairy food codes for each dairy food group
with open(data_path / 'mappings/dairy_mapping.json', 'r') as fp:
    dairy_mapping = json.load(fp)

In [ ]:
# Extract the non-compoiste dairy ingredient food codes
dairy_codes = []

for cat in dairy_categories:
  dairy_codes+=dairy_mapping[cat]['items_only']

In [ ]:
nutrients_per_gram_dairy = item_only_nutrients_dairy(item_codes=dairy_codes, nutrients=nutrients, ndb_data=ndb_data)

Add the environmental data per gram of each dairy ingredient

In [ ]:
nutrients_per_gram_dairy.loc[:, env_columns] = np.nan
nutrients_per_gram_dairy = nutrients_per_gram_dairy.apply(lambda row: add_env_data_food_code(row,
                                                                                             df_impacts = df_impacts,
                                                                                             matched_items_data = matched_items_data,
                                                                                             env_columns = env_columns,
                                                                                             mean_env_columns = mean_env_columns,
                                                                                             error_columns = error_columns),
                                                          axis=1)

Some ingredients did not share a food code with items in the environmental impact dataset. Include impacts absed on nearest neighbour matches for these data as well as the nutrient data from the 2023 NDB

In [ ]:
dairy_missing_ingredients = {613: ['Skimmed milk'],
 8544: ['Skimmed milk'],
 616: ['Skimmed milk'],
700: ['Skimmed milk'],
10251: ['Skimmed milk'],
 608: ['Semi skimmed milk'],
 8543: ['Semi skimmed milk'],
 602: ['Whole milk'],
 603: ['Whole milk'],
 604: ['Whole milk'],
 605: ['Whole milk'],
 7738: ['Natural fromage frais, fat free, sugar free'],
 7112: ['Soft cheese, full-fat (e.g. Philadelphia)',
'Soft cheese, reduced fat (e.g. Philadelphia Light/Extra Light)'],
 659: ['Reduced fat hard cheese (e.g. Cheddar/Cheshire)'],
 10977: ['Petits Filous fromage frais'],
 7727: ['Edam cheese'],
 654: ['Cheddar cheese'],
 688: ['Double Gloucester cheese', 'Cheddar cheese'],
 7735: ['Natural fromage frais, sugar free'],
 639: ['Single cream'],
 640: ['Single cream'],
 645: ['Whipped cream'],
 646: ['Whipped cream']}

In [ ]:
for food_code, item_match in dairy_missing_ingredients.items():
  # include the environmental impacts
  impacts = df_impacts.loc[item_match, env_columns].mean(axis=0)
  nutrients_per_gram_dairy.loc[food_code, env_columns] = impacts/100

  # include the nutrients
  nutrients_per_100g = ndb_data[ndb_data['FCT record ID']==food_code][nutrients].mean(axis=0)
  # Use NDB 2023 data if 2022 data are unavailable
  if nutrients_per_100g.isna().all():
    nutrients_per_100g = ndb_comprehensive[ndb_comprehensive['FCT record ID']==food_code][nutrients].mean(axis=0)

  nutrients_per_gram_dairy.loc[food_code, nutrients] = nutrients_per_100g/100

In [ ]:
for nutr in nutrients:
  nutrients_per_gram_dairy.loc[10977, nutr] = float(ndb_data[ndb_data['FCT record ID']==10218][nutr])/100

for col in env_columns:
  nutrients_per_gram_dairy.loc[10977, col] = df_impacts.loc['Petits Filous fromage frais', col]/100

/tmp/ipython-input-1795773493.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  nutrients_per_gram_dairy.loc[10977, nutr] = float(ndb_data[ndb_data['FCT record ID']==10218][nutr])/100
/tmp/ipython-input-1795773493.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  nutrients_per_gram_dairy.loc[10977, nutr] = float(ndb_data[ndb_data['FCT record ID']==10218][nutr])/100
/tmp/ipython-input-1795773493.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  nutrients_per_gram_dairy.loc[10977, nutr] = float(ndb_data[ndb_data['FCT record ID']==10218][nutr])/100
/tmp/ipython-input-1795773493.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the futur

In [ ]:
nutrients_per_gram_dairy['Calciummg'].isnull().sum()

np.int64(0)

In [ ]:
# Save the dataset of nutrients and impacts per gram of each dairy item
save_path = data_path / 'nutrients_per_gram_dairy.parquet'
nutrients_per_gram_dairy.to_parquet(save_path)